In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q wandb==0.16.6
import os
os.kill(os.getpid(), 9)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 83.9 MB/s eta 0:00:00


In [1]:
from unsloth import FastLanguageModel
print("✅ Unsloth OK")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ Unsloth OK


In [3]:
#carregar fata set#
import json, urllib.request
from datasets import Dataset

BASE_URL = 'https://raw.githubusercontent.com/Joao-Matheus-Amorim/pepe-ai/main/training/datasets/'

def load_jsonl_from_url(url):
    with urllib.request.urlopen(url) as r:
        return [json.loads(line) for line in r.read().decode().splitlines() if line.strip()]

train_data = load_jsonl_from_url(BASE_URL + 'pepe_train.jsonl')
val_data   = load_jsonl_from_url(BASE_URL + 'pepe_val.jsonl')

print(f'✅ Treino:    {len(train_data)} exemplos')
print(f'✅ Validação: {len(val_data)} exemplos')

✅ Treino:    243 exemplos
✅ Validação: 28 exemplos


In [6]:
!pip install -q "wandb>=0.17" "protobuf>=4.25,<5"
import importlib, wandb
importlib.reload(wandb)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 20.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2024.11.4 requires protobuf<4.0.0, but you have protobuf 4.25.9 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.9 which is incompatible.


ImportError: cannot import name 'Imports' from 'wandb.proto.wandb_telemetry_pb2' (/usr/local/lib/python3.12/dist-packages/wandb/proto/wandb_telemetry_pb2.py)

# 🤖 Pepê AI — Fine-tuning com LoRA

Fine-tuning do **Llama 3 8B** com os dados do Pepê usando **QLoRA** via Unsloth.

**Requisitos:** Runtime → Alterar tipo de ambiente de execução → **GPU T4**


## ⚙️ Etapa 1 — Instalar dependências e corrigir conflitos

In [ ]:
# Verifica GPU
!nvidia-smi

# Corrige conflito protobuf/wandb ANTES de instalar Unsloth
!pip install -q protobuf==4.25.3 wandb --upgrade

# Instala Unsloth com versões compatíveis
!pip install -q unsloth==2024.11.4
!pip install -q "datasets>=2.16" "transformers>=4.40" accelerate bitsandbytes trl huggingface_hub

# Reinicia runtime para aplicar correção de protobuf
print('✅ Dependências instaladas. Reiniciando runtime...')
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

> ⚠️ **Após o runtime reiniciar**, continue da Etapa 2 em diante (não re-execute a Etapa 1).

## 📦 Etapa 2 — Carregar dataset do GitHub

In [3]:
import json
import urllib.request
from datasets import Dataset

BASE_URL = 'https://raw.githubusercontent.com/Joao-Matheus-Amorim/pepe-ai/main/training/datasets/'

def load_jsonl_from_url(url):
    with urllib.request.urlopen(url) as r:
        return [json.loads(line) for line in r.read().decode().splitlines() if line.strip()]

train_data = load_jsonl_from_url(BASE_URL + 'pepe_train.jsonl')
val_data   = load_jsonl_from_url(BASE_URL + 'pepe_val.jsonl')

print(f'✅ Treino:    {len(train_data)} exemplos')
print(f'✅ Validação: {len(val_data)} exemplos')
print('\n--- Exemplo ---')
for msg in train_data[0]['messages']:
    role = msg['role'].upper()
    content = msg.get('content') or '[tool_call]'
    print(f'[{role}] {content[:100]}')

✅ Treino:    243 exemplos
✅ Validação: 28 exemplos

--- Exemplo ---
[SYSTEM] Você é Pepê, um agente de IA pessoal inteligente, direto e eficiente. Você tem acesso a ferramentas 
[USER] Meu nome é João Matheus e trabalho com Python.
[ASSISTANT] Olá, João Matheus! Ótimo ter você aqui. Posso te ajudar com Python e qualquer outra tarefa. O que pr
[USER] Quero criar um agente de IA.
[ASSISTANT] Boa escolha, João Matheus! Para criar um agente de IA em Python, recomendo usar LangChain com LangGr
[USER] Estou usando Ollama com llama3.1.
[ASSISTANT] Perfeito! Com Ollama + llama3.1 você tem uma base sólida. Vou focar os exemplos nessa stack para mon


## 🧠 Etapa 3 — Carregar Llama 3 8B + QLoRA

In [6]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'unsloth/llama-3-8b-Instruct-bnb-4bit',
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)
print(f'✅ Modelo: {model.num_parameters()/1e6:.0f}M parâmetros')

==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✅ Modelo: 8030M parâmetros


In [4]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'unsloth/Meta-Llama-3-8B-Instruct',
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)

print(f'✅ Modelo carregado: {model.num_parameters()/1e6:.0f}M parâmetros')

RuntimeError: Unsloth: No config file found - are you sure the `model_name` is correct?
If you're using a model on your local device, confirm if the folder location exists.
If you're using a HuggingFace online model, check if it exists.

In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha     = 32,
    lora_dropout   = 0.05,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'✅ LoRA: {trainable/1e6:.2f}M treináveis ({100*trainable/total:.2f}%)')

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.6 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✅ LoRA: 41.94M treináveis (0.92%)


## 📝 Etapa 4 — Formatar dataset e treinar

In [8]:
from unsloth.chat_templates import get_chat_template
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

tokenizer = get_chat_template(tokenizer, chat_template='llama-3')

def format_messages(examples):
    texts = []
    for msgs in examples['messages']:
        clean = [m for m in msgs
                 if m.get('role') in ('system', 'user', 'assistant')
                 and m.get('content') is not None]
        texts.append(tokenizer.apply_chat_template(
            clean, tokenize=False, add_generation_prompt=False
        ))
    return {'text': texts}

train_hf = Dataset.from_list(train_data).map(format_messages, batched=True)
val_hf   = Dataset.from_list(val_data).map(format_messages, batched=True)
print(f'✅ Dataset formatado: {len(train_hf)} treino / {len(val_hf)} val')

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_hf,
    eval_dataset       = val_hf,
    dataset_text_field = 'text',
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,
    packing            = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs            = 3,
        warmup_steps                = 10,
        learning_rate               = 2e-4,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 10,
        eval_strategy               = 'steps',
        eval_steps                  = 50,
        save_strategy               = 'steps',
        save_steps                  = 100,
        output_dir                  = '/content/pepe-ft-checkpoints',
        optim                       = 'adamw_8bit',
        weight_decay                = 0.01,
        lr_scheduler_type           = 'cosine',
        seed                        = 42,
        report_to                   = 'none',
    ),
)

print('🚀 Iniciando treinamento...')
stats = trainer.train()
print(f'\n✅ Concluído em {stats.metrics["train_runtime"]:.0f}s — loss: {stats.metrics["train_loss"]:.4f}')

Map:   0%|          | 0/243 [00:00<?, ? examples/s]

Map:   0%|          | 0/28 [00:00<?, ? examples/s]

✅ Dataset formatado: 243 treino / 28 val


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_hf,
    eval_dataset       = val_hf,
    dataset_text_field = 'text',
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,
    packing            = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs            = 3,
        warmup_steps                = 10,
        learning_rate               = 2e-4,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 10,
        evaluation_strategy         = 'steps',
        eval_steps                  = 50,
        save_strategy               = 'steps',
        save_steps                  = 100,
        output_dir                  = '/content/pepe-ft-checkpoints',
        optim                       = 'adamw_8bit',
        weight_decay                = 0.01,
        lr_scheduler_type           = 'cosine',
        seed                        = 42,
        report_to                   = 'none',
    ),
)

print('🚀 Iniciando treinamento...')
stats = trainer.train()
print(f'\n✅ Concluído em {stats.metrics["train_runtime"]:.0f}s — loss: {stats.metrics["train_loss"]:.4f}')

## 💾 Etapa 5 — Exportar GGUF e baixar

In [ ]:
model.save_pretrained('/content/pepe-ft-lora')
tokenizer.save_pretrained('/content/pepe-ft-lora')
print('✅ Adaptadores LoRA salvos')

print('🔄 Exportando GGUF Q4_K_M...')
model.save_pretrained_gguf(
    '/content/pepe-ft-gguf',
    tokenizer,
    quantization_method='q4_k_m'
)

import os
for f in os.listdir('/content/pepe-ft-gguf'):
    size = os.path.getsize(f'/content/pepe-ft-gguf/{f}') / 1e9
    print(f'  {f}: {size:.2f} GB')

In [ ]:
from google.colab import files
import os

for fname in os.listdir('/content/pepe-ft-gguf'):
    if fname.endswith('.gguf'):
        print(f'📥 Baixando {fname}...')
        files.download(f'/content/pepe-ft-gguf/{fname}')
        print('✅ Download iniciado')

## 🧪 Etapa 6 — Testar o modelo

In [ ]:
FastLanguageModel.for_inference(model)

SYSTEM_PEPE = (
    'Você é Pepê, um agente de IA pessoal inteligente, direto e eficiente. '
    'Você tem acesso a ferramentas de busca web, clima, visão de tela, execução de comandos, '
    'leitura de arquivos e memória persistente. '
    'Responda sempre em português do Brasil, de forma clara e concisa.'
)

def testar(pergunta):
    msgs = [
        {'role': 'system', 'content': SYSTEM_PEPE},
        {'role': 'user',   'content': pergunta},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    out = model.generate(input_ids=inputs, max_new_tokens=256,
                         temperature=0.7, do_sample=True)
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    return decoded.split('assistant')[-1].strip()

perguntas = [
    'Quem é você?',
    'Quem criou você?',
    'Qual a stack do pepe-ai?',
    'O que você consegue fazer?',
    'Qual o próximo passo do projeto?',
]

for p in perguntas:
    print(f'\n[USER] {p}')
    print(f'[PEPÊ] {testar(p)}')

## 🚀 Etapa 7 — Integrar com Ollama no seu PC

Após baixar o `.gguf`, rode no **PowerShell do seu PC**:

```powershell
# Crie o Modelfile na pasta onde salvou o .gguf
cd E:\pepe-ai

# Crie o arquivo Modelfile
@'
FROM ./pepe-ft-unsloth.Q4_K_M.gguf
SYSTEM "Você é Pepê, agente pessoal do João Matheus. Direto, brasileiro, competente."
PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER num_ctx 4096
'@ | Out-File -Encoding utf8 Modelfile

# Cria e testa
ollama create pepe-ft -f Modelfile
ollama run pepe-ft "Quem é você?"
```

Depois edite o `.env` do projeto:
```
PEPE_MODEL_PROVIDER=ollama
PEPE_MODEL_NAME=pepe-ft
```
